### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_XJIEFWXGORXK


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250522_squid_cal520"
recname = "rec5"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec5"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_transformed_corrected_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_transformed_corrected_renamed/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(717000,) (10909,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(115, 717000)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.388311,0.114731,0.109056,0.237690,0.029783,-0.258555,-0.054906,-0.127943,...,-0.205540,-0.246411,-0.135841,-0.313669,-0.470443,-0.115640,0.248679,-0.089756,-0.347672,-0.052105
1,0,1,0.146311,-0.258506,0.025748,-0.063502,-0.475334,-0.491816,-0.315671,0.105383,...,0.139565,0.366510,0.614493,0.625464,0.620554,0.683682,0.473379,0.404571,1.027857,0.647128
2,0,2,0.175034,0.405564,0.062691,-0.042655,-0.294497,-0.385188,0.101973,0.259930,...,-0.141476,-0.263522,-0.338727,-0.277598,-0.170478,-0.192374,-0.390411,-0.315076,-0.392809,-0.110839
3,0,3,0.291387,-0.584785,-0.278059,-0.037710,0.103370,0.152172,0.092864,-0.236892,...,-0.884691,-0.664574,-0.758803,-0.904036,-0.810809,-0.888937,-0.781334,-0.807268,-1.058984,-0.569531
4,0,4,0.475864,-0.249548,-0.177237,0.020654,-0.076612,-0.225005,-0.134237,0.040852,...,0.309146,0.011412,-0.081622,0.211314,-0.139862,0.159847,0.272186,0.021116,-0.037100,0.149832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,0,110,0.290092,0.295601,0.608728,0.553388,0.212037,0.287920,0.255284,-0.003236,...,-0.438539,-0.177077,-0.035762,-0.101873,-0.190090,-0.503941,-0.259739,-0.180314,0.046878,0.108681
111,0,111,0.265814,0.478750,0.478419,0.483831,0.442781,0.048049,-0.083486,0.170342,...,-0.320793,-0.238688,-0.377566,-0.465970,-0.419428,-0.408404,-0.516457,-0.427151,-0.182336,-0.245391
112,0,112,0.352052,0.158302,-0.003994,-0.091604,-0.084954,-0.204477,-0.409322,-0.125086,...,-0.324420,0.187154,0.214052,-0.053315,0.279334,0.581954,0.617110,0.196878,0.151494,0.195365
113,0,113,0.614186,-0.451197,-0.563826,-0.388242,-0.207275,-0.258426,-0.179100,-0.356985,...,-0.625112,-0.610004,-0.871106,-0.945718,-0.785358,-0.823805,-0.732217,-0.888980,-1.100406,-1.061577


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec5"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_XJIEFWXGORXK/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec5"


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_XJIEFWXGORXK/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec5"', returncode=0)